# 21.5 Scaled Dot-Product Attention 🧠⚡

We now have all the pieces:

- **Query \(Q\)** → what am I looking for?
- **Key \(K\)** → how relevant is each position?
- **Value \(V\)** → what information should I retrieve?
- **Attention score** → Query–Key compatibility
- **Softmax** → converts scores into normalized weights

The complete mechanism is:

$$
\boxed{
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
}
$$

This is the **Scaled Dot-Product Attention** formulation introduced in *Attention Is All You Need*.

---

# 1. First understand the equation as a pipeline

Don't memorize the equation as one giant expression.

Read it **left → right**:

$$
\boxed{
QK^T
}
$$

means:

> **Compare every Query with every Key.**

Then:

$$
\boxed{
\frac{QK^T}{\sqrt{d_k}}
}
$$

means:

> **Scale the scores so their magnitude remains manageable.**

Then:

$$
\boxed{
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)
}
$$

means:

> **Convert scores into attention weights.**

Finally:

$$
\boxed{
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
}
$$

means:

> **Use those weights to take a weighted combination of the Values.**

So:

```text
Q × Kᵀ
   ↓
Raw Scores
   ↓
÷ √dₖ
   ↓
Scaled Scores
   ↓
Softmax
   ↓
Attention Weights
   ↓
× V
   ↓
Attention Output
```

This is precisely the computation described in the original Transformer paper.

---

# 2. Step 1 — Compute \(QK^T\)

Suppose:

$$
Q\in\mathbb{R}^{T_q\times d_k}
$$

and:

$$
K\in\mathbb{R}^{T_k\times d_k}
$$

We transpose \(K\):

$$
K^T\in\mathbb{R}^{d_k\times T_k}
$$

Therefore:

$$
QK^T
$$

has shape:

$$
\boxed{
(T_q\times d_k)(d_k\times T_k)
=
T_q\times T_k
}
$$

So:

$$
\boxed{
QK^T\in\mathbb{R}^{T_q\times T_k}
}
$$

This matrix contains **all Query–Key compatibility scores**.

### Example

Suppose:

$$
Q=
\begin{bmatrix}
q_1\\
q_2
\end{bmatrix}
$$

and:

$$
K=
\begin{bmatrix}
k_1\\
k_2\\
k_3
\end{bmatrix}
$$

Then:

$$
QK^T=
\begin{bmatrix}
q_1\cdot k_1 & q_1\cdot k_2 & q_1\cdot k_3\\
q_2\cdot k_1 & q_2\cdot k_2 & q_2\cdot k_3
\end{bmatrix}
$$

So each row answers:

> **For this Query, how relevant is every Key?**

---

# 3. Why \(K^T\)?

This is a very common implementation question.

We have:

$$
Q:
$$

$$
[T_q,d_k]
$$

and:

$$
K:
$$

$$
[T_k,d_k]
$$

We want to calculate dot products between the Query vectors and Key vectors.

Therefore we transpose \(K\):

$$
K^T:
$$

$$
[d_k,T_k]
$$

so matrix multiplication becomes possible:

$$
[T_q,d_k]
\times
[d_k,T_k]
$$

giving:

$$
[T_q,T_k]
$$

### Remember

$$
\boxed{
QK^T = \text{all Query–Key dot products}
}
$$

---

# 4. Step 2 — Why divide by \(\sqrt{d_k}\)? 📐

This is the **"scaled"** part of Scaled Dot-Product Attention.

We calculate:

$$
\boxed{
S=
\frac{QK^T}{\sqrt{d_k}}
}
$$

But why?

Suppose the components of \(q\) and \(k\) are approximately independent, have mean \(0\), and variance \(1\).

The dot product is:

$$
q\cdot k
=
\sum_{i=1}^{d_k}q_i k_i
$$

As \(d_k\) increases, the variance of this dot product grows approximately proportional to:

$$
d_k
$$

So the raw dot products can become increasingly large in magnitude. The original Transformer paper notes that this can push softmax into regions where its gradients become extremely small. Dividing by \(\sqrt{d_k}\) counteracts this growth. 

Therefore:

$$
\boxed{
\text{Scale factor}=\frac{1}{\sqrt{d_k}}
}
$$

---

# 5. Intuition for scaling

Imagine raw scores:

$$
[2,\ 4,\ 10,\ 3]
$$

Softmax strongly favors the 10.

If scores become even larger:

$$
[20,\ 40,\ 100,\ 30]
$$

the softmax becomes extremely concentrated around the largest value.

Conceptually:

```text
Moderate scores
      ↓
   Softmax
      ↓
Useful distribution

Very large scores
      ↓
   Softmax
      ↓
Almost one-hot distribution
      ↓
Very small gradients
```

Scaling helps keep the scores in a more useful numerical range.

### Important nuance

Scaling does **not** mean:

> “Make attention scores small because small numbers are better.”

It means:

> **Control the variance/magnitude of dot products so softmax does not become unnecessarily saturated as \(d_k\) grows.**

That's the mathematically important reason.

---

# 6. Step 3 — Apply Softmax

After scaling:

$$
S=
\frac{QK^T}{\sqrt{d_k}}
$$

we apply softmax **row-wise over the Keys**:

$$
\boxed{
A=
\operatorname{softmax}(S)
}
$$

or:

$$
\boxed{
A=
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)
}
$$

Each row of \(A\) becomes an attention distribution.

For example:

$$
S=
\begin{bmatrix}
1.2&3.0&0.5\\
2.5&0.3&1.1
\end{bmatrix}
$$

After row-wise softmax, conceptually:

$$
A=
\begin{bmatrix}
0.13&0.79&0.08\\
0.72&0.08&0.20
\end{bmatrix}
$$

Notice:

$$
0.13+0.79+0.08=1
$$

and:

$$
0.72+0.08+0.20=1
$$

Each Query therefore gets its own probability distribution over the Keys.

---

# 7. Step 4 — Multiply by \(V\)

Now we have attention weights:

$$
A=
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)
$$

We multiply them by the Values:

$$
\boxed{
O=AV
}
$$

Therefore:

$$
\boxed{
O=
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
}
$$

This is the final attention output.

---

# 8. Complete Numerical Example 🧮

Let's make the mathematics concrete.

Suppose:

$$
Q=
\begin{bmatrix}
1&0
\end{bmatrix}
$$

and:

$$
K=
\begin{bmatrix}
1&0\\
0&1
\end{bmatrix}
$$

and:

$$
V=
\begin{bmatrix}
10&0\\
0&20
\end{bmatrix}
$$

Here:

$$
d_k=2
$$

### Step 1: \(QK^T\)

$$
QK^T
=
\begin{bmatrix}
1&0
\end{bmatrix}
\begin{bmatrix}
1&0\\
0&1
\end{bmatrix}
$$

Therefore:

$$
QK^T=
\begin{bmatrix}
1&0
\end{bmatrix}
$$

---

### Step 2: Scale

$$
\sqrt{d_k}=\sqrt2
$$

Therefore:

$$
\frac{QK^T}{\sqrt2}
=
\begin{bmatrix}
0.707&0
\end{bmatrix}
$$

---

### Step 3: Softmax

$$
\operatorname{softmax}([0.707,0])
\approx
[0.67,0.33]
$$

So:

$$
A=
\begin{bmatrix}
0.67&0.33
\end{bmatrix}
$$

---

### Step 4: Multiply by \(V\)

$$
O=
\begin{bmatrix}
0.67&0.33
\end{bmatrix}
\begin{bmatrix}
10&0\\
0&20
\end{bmatrix}
$$

Therefore:

$$
O=
\begin{bmatrix}
6.7&6.6
\end{bmatrix}
$$

So the output is a **weighted combination of the Values**.

---

# 9. The whole computation at once

Everything we just calculated can be compressed into:

$$
\boxed{
O=
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
}
$$

That's why this equation is so important.

You should be able to mentally expand it into:

```text
Q ──────┐
        │
        ▼
       QKᵀ
        │
        ▼
    divide √dₖ
        │
        ▼
      Softmax
        │
        ▼
  Attention weights
        │
        ├──────────► V
        │             │
        ▼             ▼
       weights × values
             │
             ▼
       Attention Output
```

---

# 10. Shape Analysis — Very Important for PyTorch 💻

Suppose:

$$
Q\in\mathbb{R}^{B\times T_q\times d_k}
$$

$$
K\in\mathbb{R}^{B\times T_k\times d_k}
$$

$$
V\in\mathbb{R}^{B\times T_k\times d_v}
$$

where:

- \(B\) = batch size
- \(T_q\) = number of Query positions
- \(T_k\) = number of Key/Value positions
- \(d_k\) = Query/Key dimension
- \(d_v\) = Value dimension

Then:

### QKᵀ

$$
(B,T_q,d_k)
\times
(B,d_k,T_k)
$$

gives:

$$
\boxed{
(B,T_q,T_k)
}
$$

These are the attention scores.

Then:

$$
\operatorname{softmax}
\rightarrow
(B,T_q,T_k)
$$

Then multiply by:

$$
V=(B,T_k,d_v)
$$

giving:

$$
(B,T_q,T_k)
\times
(B,T_k,d_v)
$$

Therefore:

$$
\boxed{
O\in\mathbb{R}^{B\times T_q\times d_v}
}
$$

### This shape chain is worth memorizing:

$$
\boxed{
QK^T:
(B,T_q,d_k)
\rightarrow
(B,T_q,T_k)
}
$$

$$
\boxed{
AV:
(B,T_q,T_k)
\rightarrow
(B,T_q,d_v)
}
$$

The original Transformer formulation similarly packs multiple queries, keys, and values into matrices and computes the attention output simultaneously. 

---

# 11. Self-Attention vs Cross-Attention

This equation works for **both**.

### Self-attention

$$
Q,K,V
\leftarrow
\text{same sequence}
$$

For example:

```text
"The cat eats fish"
        │
        ▼
       Q K V
```

### Cross-attention

$$
Q\leftarrow\text{Decoder}
$$

$$
K,V\leftarrow\text{Encoder}
$$

```text
Encoder ─────► K,V
                 │
                 ▼
              Attention
                 ▲
                 │
Decoder ─────► Q
```

The Transformer architecture uses cross-attention in decoder layers to allow decoder positions to attend over encoder outputs. 

---

# 12. Why is it called "Dot-Product"?

Because the compatibility function is:

$$
\boxed{
q\cdot k
}
$$

or, for all Queries and Keys:

$$
\boxed{
QK^T
}
$$

So:

**Dot-Product Attention**

= use the dot product to measure Query–Key compatibility.

**Scaled Dot-Product Attention**

= use the dot product **and divide by \(\sqrt{d_k}\)**.

---

# 13. Attention vs Scaled Attention

This distinction is useful:

### Dot-product attention

$$
\operatorname{softmax}(QK^T)V
$$

### Scaled dot-product attention

$$
\boxed{
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
}
$$

The Transformer uses the **scaled** version.

---

# 14. Optional Masking — Preview 👀

You'll study masking more deeply in the Transformer modules, but notice that implementations often insert a mask **before softmax**:

```text
QKᵀ
 ↓
Scale
 ↓
Mask  ← optional
 ↓
Softmax
 ↓
Weights
 ↓
× V
```

For example, the Transformer decoder uses masking so a position cannot attend to future target tokens.

Conceptually:

$$
\boxed{
\text{Mask} \rightarrow \text{Softmax}
}
$$

not:

$$
\text{Softmax}\rightarrow\text{Mask}
$$

We'll return to this when we study **causal masking and padding masks**.

---

# 15. Why this equation matters so much 🔥

This isn't just an equation you need to memorize for an exam.

It is the computational core of Transformer attention.

From this operation, you eventually build:

$$
\text{Attention}
\rightarrow
\text{Multi-Head Attention}
\rightarrow
\text{Transformer Block}
\rightarrow
\text{BERT/GPT/T5-style architectures}
$$

The original Transformer was built entirely around attention mechanisms rather than recurrence or convolution. 

---

# 🧠 The equation decoded

Whenever you see:

$$
\boxed{
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
}
$$

read it as:

> **Compare Queries with Keys → scale the scores → normalize them into weights → use those weights to retrieve information from Values.**

Or the ultra-short version:

$$
\boxed{
\textbf{Match → Scale → Normalize → Retrieve}
}
$$

---

## What you should remember 🎯

1. **\(QK^T\)** computes Query–Key compatibility scores.
2. **\(/\sqrt{d_k}\)** stabilizes the magnitude of the dot products as the key dimension grows.
3. **Softmax** converts scores into attention weights.
4. **Weights × \(V\)** produces the weighted information retrieved from the Values.
5. The complete operation is:

$$
\boxed{
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
}
$$

6. Shape-wise:

$$
\boxed{
(B,T_q,d_k)
\rightarrow
(B,T_q,T_k)
\rightarrow
(B,T_q,d_v)
}
$$

7. **Q and K determine where to attend; V determines what information is retrieved.**

---